# 05 Working Capital Impact

## 5.1 Business Objective

This notebook extends the inventory analytics workflow into a finance-facing working capital analysis.

The goal is to estimate how simulated inventory positions may translate into working capital exposure, including inventory value, stockout revenue exposure, and overstock capital exposure. The analysis is designed for portfolio demonstration and interview discussion. It does not use real company inventory balances, supplier costs, warehouse capacity, or operational recommendations.

## 5.2 Data Foundation

This notebook uses `outputs/sku_profile_classification.csv` as the starting point. That file contains SKU-level classification fields such as revenue, unit volume, demand volatility, SKU class, and recommended action.

Inventory-related fields such as current inventory, unit cost, supplier lead time, inventory risk, and warehouse strategy are simulated for portfolio demonstration. If these simulated fields are not already present in the source file, this notebook recreates them using transparent deterministic assumptions so the working capital outputs can be reproduced from the project files.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
outputs_dir = project_root / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

sku_profile_path = outputs_dir / "sku_profile_classification.csv"
sku_profile = pd.read_csv(sku_profile_path)

print("SKU profile shape:", sku_profile.shape)
print("Source file:", sku_profile_path)
sku_profile.head()

SKU profile shape: (3917, 12)
Source file: /Users/yeternalh/ecommerce-inventory-warehouse-allocation/outputs/sku_profile_classification.csv


,stock_code,total_units,total_revenue,active_months,avg_monthly_units,std_monthly_units,avg_unit_price,total_orders,description,demand_cv,sku_class,recommended_action
0,10002,860,759.89,5,172.000000,132.183584,1.045843,71,INFLATABLE POLITICAL GLOBE,0.768509,Regular,Maintain standard replenishment review
1,10080,303,119.09,7,43.285714,33.119553,0.455714,22,GROOVY CACTUS INFLATABLE,0.765138,Regular,Maintain standard replenishment review
2,10120,192,40.32,10,19.200000,15.454593,0.210000,29,DOGGY RUBBER,0.804927,Regular,Maintain standard replenishment review
3,10123C,5,3.25,2,2.500000,2.121320,0.650000,3,HEARTS WRAPPING TAPE,0.848528,Long-Tail,Limit stock and avoid excessive local warehous...
4,10124A,16,6.72,4,4.000000,0.816497,0.420000,5,SPOTS ON RED BOOKCOVER TAPE,0.204124,Long-Tail,Limit stock and avoid excessive local warehous...


## 5.3 Simulated Inventory and Cost Fields

The public UCI Online Retail dataset does not include actual inventory levels, unit costs, supplier lead times, storage volume, warehouse capacity, or fulfillment methods.

For this reason, the inventory and cost fields used below are simulated. They are included only to demonstrate how an inventory analytics workflow can be extended into finance-facing working capital analysis. These fields do not represent real company inventory data.

In [2]:
required_simulated_fields = [
    "current_inventory",
    "lead_time_days",
    "storage_volume_per_unit",
    "unit_cost",
    "avg_daily_demand",
    "safety_stock",
    "reorder_point",
    "recommended_replenishment_qty",
    "inventory_coverage_days",
    "inventory_risk",
    "warehouse_strategy",
]

missing_fields = [field for field in required_simulated_fields if field not in sku_profile.columns]

if missing_fields:
    print("Missing simulated fields detected:", missing_fields)
    print("Recreating simulated inventory fields using deterministic portfolio assumptions.")

    np.random.seed(42)

    sku_profile["current_inventory"] = np.where(
        sku_profile["sku_class"].isin(["High-Revenue Priority", "High-Turnover Stable"]),
        np.random.randint(20, 300, size=len(sku_profile)),
        np.random.randint(0, 120, size=len(sku_profile)),
    )

    sku_profile["lead_time_days"] = np.random.choice(
        [7, 14, 21, 30, 45],
        size=len(sku_profile),
        p=[0.20, 0.35, 0.25, 0.15, 0.05],
    )

    sku_profile["storage_volume_per_unit"] = np.random.uniform(
        0.1,
        3.0,
        size=len(sku_profile),
    ).round(2)

    sku_profile["unit_cost"] = (
        sku_profile["avg_unit_price"] *
        np.random.uniform(0.35, 0.65, size=len(sku_profile))
    ).round(2)

    sku_profile["avg_daily_demand"] = sku_profile["avg_monthly_units"] / 30

    sku_profile["safety_stock"] = (
        sku_profile["avg_daily_demand"] *
        sku_profile["lead_time_days"] *
        (0.25 + sku_profile["demand_cv"].clip(0, 2) * 0.25)
    ).round(0)

    sku_profile["reorder_point"] = (
        sku_profile["avg_daily_demand"] * sku_profile["lead_time_days"] +
        sku_profile["safety_stock"]
    ).round(0)

    sku_profile["recommended_replenishment_qty"] = (
        sku_profile["reorder_point"] - sku_profile["current_inventory"]
    ).clip(lower=0).round(0)

    sku_profile["inventory_coverage_days"] = np.where(
        sku_profile["avg_daily_demand"] > 0,
        sku_profile["current_inventory"] / sku_profile["avg_daily_demand"],
        np.inf,
    )

    def assign_inventory_risk(row):
        if row["current_inventory"] < row["reorder_point"]:
            return "Stockout Risk"
        if row["inventory_coverage_days"] > 180 and row["sku_class"] in ["Long-Tail", "Regular"]:
            return "Overstock Risk"
        return "Normal"

    sku_profile["inventory_risk"] = sku_profile.apply(assign_inventory_risk, axis=1)

    def assign_warehouse_strategy(row):
        if row["inventory_risk"] == "Overstock Risk":
            return "Overstock Review / Reduce Replenishment"
        if row["sku_class"] == "High-Revenue Priority":
            return "Local Warehouse Priority"
        if row["sku_class"] == "High-Turnover Stable":
            return "Stable Local Warehouse Inventory"
        if row["sku_class"] == "High-Turnover Volatile":
            return "Small-Batch Replenishment / Monitor Closely"
        if row["sku_class"] == "Long-Tail":
            return "External or Limited Stock Strategy"
        return "Standard Replenishment Review"

    sku_profile["warehouse_strategy"] = sku_profile.apply(assign_warehouse_strategy, axis=1)
else:
    print("Using simulated inventory fields already present in the source file.")

sku_profile[
    [
        "stock_code",
        "description",
        "sku_class",
        "current_inventory",
        "unit_cost",
        "inventory_risk",
        "warehouse_strategy",
    ]
].head()

Missing simulated fields detected: ['current_inventory', 'lead_time_days', 'storage_volume_per_unit', 'unit_cost', 'avg_daily_demand', 'safety_stock', 'reorder_point', 'recommended_replenishment_qty', 'inventory_coverage_days', 'inventory_risk', 'warehouse_strategy']
Recreating simulated inventory fields using deterministic portfolio assumptions.


,stock_code,description,sku_class,current_inventory,unit_cost,inventory_risk,warehouse_strategy
0,10002,INFLATABLE POLITICAL GLOBE,Regular,58,0.62,Stockout Risk,Standard Replenishment Review
1,10080,GROOVY CACTUS INFLATABLE,Regular,75,0.27,Normal,Standard Replenishment Review
2,10120,DOGGY RUBBER,Regular,48,0.09,Normal,Standard Replenishment Review
3,10123C,HEARTS WRAPPING TAPE,Long-Tail,58,0.32,Overstock Risk,Overstock Review / Reduce Replenishment
4,10124A,SPOTS ON RED BOOKCOVER TAPE,Long-Tail,30,0.26,Overstock Risk,Overstock Review / Reduce Replenishment


## 5.4 Working Capital Metrics

The finance-facing metrics are intentionally simple and transparent:

- `estimated_inventory_value`: estimated inventory investment based on simulated current inventory and simulated unit cost.
- `stockout_revenue_exposure`: estimated revenue exposure for Stockout Risk SKUs based on recommended replenishment quantity and average unit price.
- `overstock_capital_exposure`: estimated capital tied up above 180 days of demand coverage for Overstock Risk SKUs.

These measures are illustrative decision-support indicators. They are not accounting values and should not be interpreted as real company financial exposure.

In [3]:
sku_profile["estimated_inventory_value"] = (
    sku_profile["current_inventory"] * sku_profile["unit_cost"]
).round(2)

sku_profile["stockout_revenue_exposure"] = np.where(
    sku_profile["inventory_risk"] == "Stockout Risk",
    sku_profile["recommended_replenishment_qty"] * sku_profile["avg_unit_price"],
    0,
).round(2)

sku_profile["inventory_units_above_180_days"] = np.where(
    sku_profile["inventory_risk"] == "Overstock Risk",
    (
        sku_profile["current_inventory"] -
        (sku_profile["avg_daily_demand"] * 180)
    ).clip(lower=0),
    0,
).round(2)

sku_profile["overstock_capital_exposure"] = (
    sku_profile["inventory_units_above_180_days"] * sku_profile["unit_cost"]
).round(2)

sku_profile[
    [
        "stock_code",
        "description",
        "sku_class",
        "inventory_risk",
        "estimated_inventory_value",
        "stockout_revenue_exposure",
        "overstock_capital_exposure",
    ]
].head(10)

,stock_code,description,sku_class,inventory_risk,estimated_inventory_value,stockout_revenue_exposure,overstock_capital_exposure
0,10002,INFLATABLE POLITICAL GLOBE,Regular,Stockout Risk,35.96,59.61,0.00
1,10080,GROOVY CACTUS INFLATABLE,Regular,Normal,20.25,0.00,0.00
2,10120,DOGGY RUBBER,Regular,Normal,4.32,0.00,0.00
3,10123C,HEARTS WRAPPING TAPE,Long-Tail,Overstock Risk,18.56,0.00,13.76
4,10124A,SPOTS ON RED BOOKCOVER TAPE,Long-Tail,Overstock Risk,7.80,0.00,1.56
5,10124G,ARMY CAMO BOOKCOVER TAPE,Long-Tail,Overstock Risk,13.80,0.00,9.98
6,10125,MINI FUNKY DESIGN TAPES,Regular,Stockout Risk,37.40,75.45,0.00
7,10133,COLOURING PENCILS BROWN TUBE,High-Turnover Stable,Normal,45.12,0.00,0.00
8,10135,COLOURING PENCILS BROWN TUBE,High-Turnover Stable,Stockout Risk,187.20,220.39,0.00
9,11001,ASSTD DESIGN RACING CAR PEN,Regular,Normal,95.04,0.00,0.00


## 5.5 Inventory Value by SKU Class

This view helps management understand how simulated inventory value is distributed across SKU classes. It connects operational segmentation to a finance-facing view of inventory investment.

In [4]:
inventory_value_by_sku_class = (
    sku_profile
    .groupby("sku_class", as_index=False)
    .agg(
        sku_count=("stock_code", "count"),
        total_inventory_units=("current_inventory", "sum"),
        estimated_inventory_value=("estimated_inventory_value", "sum"),
        stockout_revenue_exposure=("stockout_revenue_exposure", "sum"),
        overstock_capital_exposure=("overstock_capital_exposure", "sum"),
    )
    .sort_values("estimated_inventory_value", ascending=False)
)

for col in ["estimated_inventory_value", "stockout_revenue_exposure", "overstock_capital_exposure"]:
    inventory_value_by_sku_class[col] = inventory_value_by_sku_class[col].round(2)

inventory_value_by_sku_class

,sku_class,sku_count,total_inventory_units,estimated_inventory_value,stockout_revenue_exposure,overstock_capital_exposure
0,High-Revenue Priority,784,121465,746708.10,524066.59,0.00
3,Long-Tail,1172,70163,187267.85,1685.29,120201.25
4,Regular,1684,98480,140990.12,88834.00,3924.71
1,High-Turnover Stable,208,33156,13613.81,19109.15,0.00
2,High-Turnover Volatile,69,4278,2033.19,11580.53,0.00


## 5.6 Inventory Value by Warehouse Strategy

This view connects simulated working capital exposure to warehouse strategy recommendations. It helps identify where inventory value is concentrated across local warehouse priority, standard review, external or limited-stock, and overstock review groups.

In [5]:
inventory_value_by_warehouse_strategy = (
    sku_profile
    .groupby("warehouse_strategy", as_index=False)
    .agg(
        sku_count=("stock_code", "count"),
        total_inventory_units=("current_inventory", "sum"),
        estimated_inventory_value=("estimated_inventory_value", "sum"),
        stockout_revenue_exposure=("stockout_revenue_exposure", "sum"),
        overstock_capital_exposure=("overstock_capital_exposure", "sum"),
    )
    .sort_values("estimated_inventory_value", ascending=False)
)

for col in ["estimated_inventory_value", "stockout_revenue_exposure", "overstock_capital_exposure"]:
    inventory_value_by_warehouse_strategy[col] = inventory_value_by_warehouse_strategy[col].round(2)

inventory_value_by_warehouse_strategy

,warehouse_strategy,sku_count,total_inventory_units,estimated_inventory_value,stockout_revenue_exposure,overstock_capital_exposure
1,Local Warehouse Priority,784,121465,746708.10,524066.59,0.00
2,Overstock Review / Reduce Replenishment,924,68196,185101.30,0.00,124125.96
5,Standard Replenishment Review,1597,90208,124231.13,88834.00,0.00
0,External or Limited Stock Strategy,335,10239,18925.54,1685.29,0.00
4,Stable Local Warehouse Inventory,208,33156,13613.81,19109.15,0.00
3,Small-Batch Replenishment / Monitor Closely,69,4278,2033.19,11580.53,0.00


## 5.7 Exposure Review Lists

The top exposure lists are designed for management review. They identify SKUs with the largest simulated capital exposure from overstock risk and the largest simulated revenue exposure from stockout risk.

In [6]:
top_overstock_capital_exposure = (
    sku_profile[sku_profile["overstock_capital_exposure"] > 0]
    .sort_values("overstock_capital_exposure", ascending=False)
    [
        [
            "stock_code",
            "description",
            "sku_class",
            "warehouse_strategy",
            "current_inventory",
            "inventory_coverage_days",
            "unit_cost",
            "estimated_inventory_value",
            "overstock_capital_exposure",
        ]
    ]
    .head(25)
)

top_stockout_revenue_exposure = (
    sku_profile[sku_profile["stockout_revenue_exposure"] > 0]
    .sort_values("stockout_revenue_exposure", ascending=False)
    [
        [
            "stock_code",
            "description",
            "sku_class",
            "warehouse_strategy",
            "avg_monthly_units",
            "current_inventory",
            "reorder_point",
            "recommended_replenishment_qty",
            "avg_unit_price",
            "stockout_revenue_exposure",
        ]
    ]
    .head(25)
)

print("Top overstock exposure rows:", len(top_overstock_capital_exposure))
print("Top stockout exposure rows:", len(top_stockout_revenue_exposure))

top_overstock_capital_exposure.head()

Top overstock exposure rows: 25
Top stockout exposure rows: 25


,stock_code,description,sku_class,warehouse_strategy,current_inventory,inventory_coverage_days,unit_cost,estimated_inventory_value,overstock_capital_exposure
1692,22823,CHEST NATURAL WOOD 20 DRAWERS,Long-Tail,Overstock Review / Reduce Replenishment,115,1437.5,73.52,8454.80,7396.11
3244,84963B,BLUE PAINTED KASHMIRI CHAIR,Long-Tail,Overstock Review / Reduce Replenishment,117,1170.0,24.01,2809.17,2376.99
3915,gift_0001_50,DOTCOMGIFTSHOP GIFT VOUCHER £50.00,Long-Tail,Overstock Review / Reduce Replenishment,86,1935.0,21.50,1849.00,1677.00
3913,gift_0001_30,DOTCOMGIFTSHOP GIFT VOUCHER £30.00,Long-Tail,Overstock Review / Reduce Replenishment,105,2250.0,16.19,1699.95,1563.95
3910,S,SAMPLES,Long-Tail,Overstock Review / Reduce Replenishment,97,2910.0,15.88,1540.36,1445.08


## 5.8 Management Summary Output

The final working capital summary consolidates the most important finance-facing indicators for interview discussion and management review.

In [7]:
summary_values = {
    "total_skus": str(len(sku_profile)),
    "total_estimated_inventory_value": f"{sku_profile['estimated_inventory_value'].sum():.2f}",
    "stockout_risk_skus": str(int((sku_profile["inventory_risk"] == "Stockout Risk").sum())),
    "stockout_revenue_exposure": f"{sku_profile['stockout_revenue_exposure'].sum():.2f}",
    "overstock_risk_skus": str(int((sku_profile["inventory_risk"] == "Overstock Risk").sum())),
    "overstock_capital_exposure": f"{sku_profile['overstock_capital_exposure'].sum():.2f}",
    "sku_classes": str(sku_profile["sku_class"].nunique()),
    "warehouse_strategy_groups": str(sku_profile["warehouse_strategy"].nunique()),
}

working_capital_summary = pd.DataFrame(
    {
        "metric": list(summary_values.keys()),
        "value": list(summary_values.values()),
    }
)

working_capital_summary

,metric,value
0,total_skus,3917
1,total_estimated_inventory_value,1090613.07
2,stockout_risk_skus,1432
3,stockout_revenue_exposure,645275.56
4,overstock_risk_skus,924
5,overstock_capital_exposure,124125.96
6,sku_classes,5
7,warehouse_strategy_groups,6


## 5.9 Save Management Outputs

This notebook saves five new working-capital output files. It does not modify the existing project outputs generated by notebooks 01 through 04.

In [8]:
working_capital_summary.to_csv(outputs_dir / "working_capital_summary.csv", index=False)
top_overstock_capital_exposure.to_csv(outputs_dir / "top_overstock_capital_exposure.csv", index=False)
top_stockout_revenue_exposure.to_csv(outputs_dir / "top_stockout_revenue_exposure.csv", index=False)
inventory_value_by_sku_class.to_csv(outputs_dir / "inventory_value_by_sku_class.csv", index=False)
inventory_value_by_warehouse_strategy.to_csv(outputs_dir / "inventory_value_by_warehouse_strategy.csv", index=False)

print("Saved output files:")
print("- outputs/working_capital_summary.csv")
print("- outputs/top_overstock_capital_exposure.csv")
print("- outputs/top_stockout_revenue_exposure.csv")
print("- outputs/inventory_value_by_sku_class.csv")
print("- outputs/inventory_value_by_warehouse_strategy.csv")

Saved output files:
- outputs/working_capital_summary.csv
- outputs/top_overstock_capital_exposure.csv
- outputs/top_stockout_revenue_exposure.csv
- outputs/inventory_value_by_sku_class.csv
- outputs/inventory_value_by_warehouse_strategy.csv


## 5.10 Management Interpretation

This working capital extension shows how inventory analytics can support finance-facing discussion. High estimated inventory value may indicate where working capital is concentrated, stockout revenue exposure highlights SKUs that may need replenishment attention, and overstock capital exposure highlights SKUs where simulated inventory may be tying up capital beyond demand needs.

The outputs are designed as decision-support examples for portfolio discussion. They do not represent real inventory balances, actual supplier costs, or actual company financial exposure.